In [ ]:
import numpy as np

# 1. Define the multivariable function, its gradient, and Hessian
def function_f(x):
    x1, x2 = x
    return (1/3)*x1**3 + x1*x2 + 0.5*x2**2 + 2*x2 - (2/3)

def grad_f(x):
    x1, x2 = x
    return np.array([x1**2 + x2, x1 + x2 + 2])

def hessian_f(x):
    x1, x2 = x
    return np.array([[2*x1, 1.0], [1.0, 1.0]])

# 2. Reusable stationary point finder (Newton's Method root-finder for grad = 0)
def find_stationary_points(x0, tol=1e-6, max_iter=100):
    x = np.array(x0, dtype=float)
    for k in range(max_iter):
        g = grad_f(x)
        if np.linalg.norm(g) < tol:
            return x, k
        
        H = hessian_f(x)
        try:
            # Newton's step direction: H * d = -g
            d = np.linalg.solve(H, -g)
        except np.linalg.LinAlgError:
            print(f"Hessian singular at iteration {k} from initial guess {x0}")
            return None, k
            
        x += d
    return x, max_iter

# 3. Reusable SOSC classifier based on Hessian eigenvalues
def classify_stationary_point(x_star, tol=1e-6):
    H = hessian_f(x_star)
    # Compute eigenvalues for symmetric matrices
    eigenvalues = np.linalg.eigvalsh(H)
    
    if np.all(eigenvalues > tol):
        label = "local minimizer (Positive Definite)"
    elif np.all(eigenvalues < -tol):
        label = "local maximizer (Negative Definite)"
    elif np.any(eigenvalues > tol) and np.any(eigenvalues < -tol):
        label = "saddle point (Indefinite)"
    else:
        label = "inconclusive (Semi-definite)"
        
    return label, eigenvalues, H



### Run Question 4(a): Multi-start Solver Simulation


In [ ]:
initial_guesses = [
    [0.0, 0.0],
    [-2.0, 0.0],
    [3.0, -3.0],
    [1.0, -5.0],
    [5.0, 5.0]
]

print("--- QUESTION 4(a) RESULTS ---")
unique_points = []
dedupe_tol = 1e-3

for x0 in initial_guesses:
    result = find_stationary_points(x0)
    if result[0] is not None:
        x_star, iters = result
        print(f"Initial Guess {x0} -> Converged to {np.round(x_star, 4)} in {iters} iterations")
        
        # Deduplicate to find unique stationary points
        if not any(np.linalg.norm(x_star - p) < dedupe_tol for p in unique_points):
            unique_points.append(x_star)



### Run Question 4(b): Second-Order Definiteness Classification


In [ ]:
print("\n--- QUESTION 4(b) RESULTS ---")
for i, x_star in enumerate(unique_points):
    label, eigs, H = classify_stationary_point(x_star)
    print(f"\nStationary Point {i+1}: {np.round(x_star, 4)}")
    print(f"  Hessian Matrix:\n{H}")
    print(f"  Eigenvalues: {np.round(eigs, 4)}")
    print(f"  Nature: {label}")
